# 04 - Quality Gate + Lineage (Rubric item 5 / 15 pts)

**Project:** ShopSense | **Program:** SDAIA Academy - Modern Data Engineering for AI Systems

This notebook builds the two things the orchestrator will use tomorrow's DAG to enforce:
a **Great Expectations** gate that stops the pipeline on bad data, and **OpenLineage**
events emitted for every stage.

## What this notebook must prove
| Rubric requirement | Where it is proven |
|---|---|
| Great Expectations checks that **actually gate** the pipeline | Section 3 - `gate()` raises `QualityGateFailed`; section 5 shows it firing on real bad data |
| The real library, not a hand-rolled validator | `great-expectations 1.x`, `openlineage-python 1.x` |
| OpenLineage **START / COMPLETE / FAIL** events **per stage** | Section 4 + section 6 - events written to `reports/lineage_events.jsonl` |
| Failure path proven | Section 5 (gate rejects) and section 6 (FAIL event with the error message facet) |

Everything is written to `src/` on Drive so notebook 05's Airflow DAG imports **the same code**,
not a copy of it.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q "great-expectations==1.22.0" "openlineage-python==1.53.0" "deltalake>=1.0" pandas pyarrow "pydantic>=2.7"
import great_expectations as gx, openlineage, deltalake, pydantic
print('great_expectations', gx.__version__)
print('deltalake        ', deltalake.__version__)
print('pydantic         ', pydantic.VERSION)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.9 MB/s eta 0:00:00
great_expectations 1.22.0
deltalake         1.6.3
pydantic          2.13.4


In [3]:
from pathlib import Path
import json, sys, datetime

PROJECT = Path('/content/drive/MyDrive/sdaia_capstone')
SRC     = PROJECT / 'src'
REPORTS = PROJECT / 'reports'
LANDING = PROJECT / 'data' / 'bronze_landing'

SRC.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
(SRC / '__init__.py').write_text('')

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

landing_files = sorted(LANDING.glob('orders_valid_*.jsonl'))
assert landing_files, f'No landing file in {LANDING} - run notebook 01 first.'
LANDING_FILE = landing_files[-1]
print('src package ->', SRC)
print('input file  ->', LANDING_FILE.name)

src package -> /content/drive/MyDrive/sdaia_capstone/src
input file  -> orders_valid_20260908T194115Z.jsonl


## 2. The transformation module

The medallion logic the DAG will call. It uses **delta-rs** (`deltalake`) rather than Spark so the
scheduler stays light - notebook 02 is the Spark/`delta-spark` implementation of the same layers,
and the rubric credits either library. The DAG writes to its own root, `lakehouse_dag/`, so it never
races with the tables notebook 02 built.

In [4]:
%%writefile /content/drive/MyDrive/sdaia_capstone/src/transforms.py
# Medallion transformations for the orchestrated pipeline (delta-rs implementation).
import json
import os
from pathlib import Path
import pandas as pd
from deltalake import DeltaTable, write_deltalake

PROJECT   = Path('/content/drive/MyDrive/sdaia_capstone')
# Delta tables are written to LOCAL disk. delta-rs commits by atomic rename, and the
# Google Drive FUSE mount does not support rename -> "Operation not permitted (os error 1)".
# Nothing is lost: every DAG run rebuilds Bronze from the landing file in Drive.
LAKE      = Path(os.environ.get('SHOPSENSE_LAKE', '/content/lakehouse_dag'))
BRONZE    = str(LAKE / 'bronze' / 'orders')
SILVER    = str(LAKE / 'silver' / 'orders')
GOLD      = str(LAKE / 'gold' / 'daily_category_revenue')
REVENUE_STATUSES = ['paid', 'shipped', 'delivered']


def latest_landing_file(override: str | None = None) -> Path:
    # An override lets the notebook point the DAG at a deliberately corrupted file
    # so the failure path can be demonstrated.
    if override:
        return Path(override)
    files = sorted((PROJECT / 'data' / 'bronze_landing').glob('orders_valid_*.jsonl'))
    if not files:
        raise FileNotFoundError('no bronze landing file - run notebook 01 first')
    return files[-1]


def read_bronze() -> pd.DataFrame:
    return DeltaTable(BRONZE).to_pandas()


def read_silver() -> pd.DataFrame:
    return DeltaTable(SILVER).to_pandas()


def ingest_to_bronze(landing_override: str | None = None) -> dict:
    src = latest_landing_file(landing_override)
    df = pd.read_json(src, lines=True)
    LAKE.mkdir(parents=True, exist_ok=True)
    write_deltalake(BRONZE, df, mode='overwrite', schema_mode='overwrite')
    return {'source_file': src.name, 'rows': int(len(df)),
            'distinct_order_id': int(df['order_id'].nunique())}


def to_silver_shape(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['order_ts']    = pd.to_datetime(out['order_ts'], format='mixed', utc=True)
    out['ingested_at'] = pd.to_datetime(out['ingested_at'], format='mixed', utc=True)
    out['order_date']  = out['order_ts'].dt.date.astype(str)
    out['line_total']  = (out['quantity'] * out['unit_price']).round(2)
    out['is_revenue']  = out['status'].isin(REVENUE_STATUSES)
    out['city']        = out['city'].str.strip().str.title()
    out['category']    = out['category'].str.strip().str.lower()
    cols = ['order_id','customer_id','customer_email','product_id','product_name','category',
            'quantity','unit_price','line_total','currency','city','status','is_revenue',
            'order_date','ingested_at']
    return out[cols]


def build_silver(landing_override: str | None = None) -> dict:
    shaped = to_silver_shape(read_bronze())
    # MERGE refuses an ambiguous source, so pick a winner per business key first:
    # the most recently ingested version of each order.
    deduped = (shaped.sort_values('ingested_at')
                     .drop_duplicates('order_id', keep='last')
                     .reset_index(drop=True))
    deduped['ingested_at'] = deduped['ingested_at'].dt.tz_localize(None)

    if not Path(SILVER, '_delta_log').exists():
        write_deltalake(SILVER, deduped, mode='overwrite', schema_mode='overwrite')
        return {'created': True, 'rows': int(len(deduped)),
                'inserted': int(len(deduped)), 'updated': 0}

    dt = DeltaTable(SILVER)
    metrics = (dt.merge(source=deduped,
                        predicate='target.order_id = source.order_id',
                        source_alias='source', target_alias='target')
                 .when_matched_update_all()
                 .when_not_matched_insert_all()
                 .execute())
    return {'created': False, 'rows': int(len(DeltaTable(SILVER).to_pandas())),
            'inserted': int(metrics.get('num_target_rows_inserted', 0)),
            'updated':  int(metrics.get('num_target_rows_updated', 0))}


def build_gold() -> dict:
    silver = read_silver()
    revenue = silver[silver['is_revenue']]
    gold = (revenue.groupby(['order_date', 'category', 'city'], as_index=False)
                   .agg(orders_count=('order_id', 'nunique'),
                        unique_customers=('customer_id', 'nunique'),
                        units_sold=('quantity', 'sum'),
                        total_revenue_sar=('line_total', 'sum'),
                        avg_order_value_sar=('line_total', 'mean')))
    gold['total_revenue_sar']   = gold['total_revenue_sar'].round(2)
    gold['avg_order_value_sar'] = gold['avg_order_value_sar'].round(2)
    write_deltalake(GOLD, gold, mode='overwrite', schema_mode='overwrite')
    return {'silver_rows': int(len(silver)), 'gold_rows': int(len(gold)),
            'total_revenue_sar': float(gold['total_revenue_sar'].sum())}


def refresh_rag_index() -> dict:
    # Re-chunks the knowledge base and writes a manifest. The embedding refresh itself
    # lives in notebook 03; this task proves the stage is wired into the DAG.
    import re
    docs = sorted((PROJECT / 'data' / 'knowledge_base').glob('*.md'))
    if not docs:
        raise FileNotFoundError('knowledge base is empty - run notebook 03 first')
    sent = re.compile(r'(?<=[.!?])\s+')
    manifest, total = [], 0
    for d in docs:
        text = re.sub(r'\s+', ' ', d.read_text()).strip()
        chunks, cur = [], ''
        for s in [x.strip() for x in sent.split(text) if x.strip()]:
            if cur and len(cur) + len(s) + 1 > 420:
                chunks.append(cur); cur = s
            else:
                cur = (cur + ' ' + s).strip()
        if cur:
            chunks.append(cur)
        manifest.append({'doc': d.name, 'chunks': len(chunks)})
        total += len(chunks)
    out = PROJECT / 'reports' / 'rag_index_manifest.json'
    out.write_text(json.dumps({'documents': len(docs), 'chunks': total,
                               'per_document': manifest}, indent=2))
    return {'documents': len(docs), 'chunks': total}

Overwriting /content/drive/MyDrive/sdaia_capstone/src/transforms.py


## 3. The quality gate (Great Expectations)

Two suites, one per layer. `gate()` runs a suite and **raises** when it fails - that raise is what
stops the Airflow DAG. A gate that only logs a warning is not a gate.

In [5]:
%%writefile /content/drive/MyDrive/sdaia_capstone/src/quality.py
# Great Expectations suites that gate the ShopSense pipeline.
import great_expectations as gx
from great_expectations import expectations as gxe

ALLOWED_CATEGORIES = ['electronics', 'grocery', 'fashion', 'home', 'beauty', 'sports']
ALLOWED_STATUSES   = ['created', 'paid', 'shipped', 'delivered', 'cancelled']
ALLOWED_CURRENCIES = ['SAR', 'USD', 'AED']


class QualityGateFailed(Exception):
    # Raised when a suite fails. Airflow turns this into a failed task,
    # which puts every downstream task into upstream_failed.
    def __init__(self, layer, failures, summary):
        self.layer, self.failures, self.summary = layer, failures, summary
        lines = '\n'.join(f'  - {f["expectation"]} on {f["column"]}: '
                          f'{f["unexpected_count"]} unexpected value(s)' for f in failures)
        super().__init__(f'Quality gate FAILED on {layer}: '
                         f'{len(failures)}/{summary["evaluated"]} expectations failed\n{lines}')


def bronze_expectations():
    return [
        gxe.ExpectColumnValuesToNotBeNull(column='order_id'),
        gxe.ExpectColumnValuesToNotBeNull(column='customer_id'),
        gxe.ExpectColumnValuesToBeBetween(column='quantity', min_value=1, max_value=100),
        gxe.ExpectColumnValuesToBeBetween(column='unit_price', min_value=0.01),
        gxe.ExpectColumnValuesToBeInSet(column='currency', value_set=ALLOWED_CURRENCIES),
        gxe.ExpectColumnValuesToBeInSet(column='status',   value_set=ALLOWED_STATUSES),
        gxe.ExpectColumnValuesToBeInSet(column='category', value_set=ALLOWED_CATEGORIES),
        gxe.ExpectTableRowCountToBeBetween(min_value=1),
    ]


def silver_expectations():
    return [
        gxe.ExpectColumnValuesToBeUnique(column='order_id'),      # the MERGE must hold the key unique
        gxe.ExpectColumnValuesToNotBeNull(column='order_id'),
        gxe.ExpectColumnValuesToNotBeNull(column='customer_id'),
        gxe.ExpectColumnValuesToBeBetween(column='line_total', min_value=0.01),
        gxe.ExpectColumnValuesToBeInSet(column='status', value_set=ALLOWED_STATUSES),
        gxe.ExpectTableRowCountToBeBetween(min_value=1),
    ]


SUITES = {'bronze': bronze_expectations, 'silver': silver_expectations}


def validate(df, layer):
    # Runs the suite for `layer` against a pandas DataFrame and returns (success, failures, summary).
    expectations = SUITES[layer]()
    ctx    = gx.get_context(mode='ephemeral')
    source = ctx.data_sources.add_pandas(f'{layer}_source')
    asset  = source.add_dataframe_asset(name=f'{layer}_orders')
    batch  = asset.add_batch_definition_whole_dataframe('whole_dataframe')

    suite = ctx.suites.add(gx.ExpectationSuite(name=f'{layer}_suite'))
    for exp in expectations:
        suite.add_expectation(exp)

    vdef = ctx.validation_definitions.add(
        gx.ValidationDefinition(name=f'{layer}_validation', data=batch, suite=suite))
    result = vdef.run(batch_parameters={'dataframe': df})

    failures, checks = [], []
    for r in result.results:
        cfg = r.expectation_config
        row = {'expectation': cfg.type,
               'column': cfg.kwargs.get('column', '(table)'),
               'success': bool(r.success),
               'unexpected_count': int(r.result.get('unexpected_count', 0) or 0)}
        checks.append(row)
        if not r.success:
            failures.append(row)

    summary = {'layer': layer, 'rows': int(len(df)), 'evaluated': len(checks),
               'passed': len(checks) - len(failures), 'failed': len(failures),
               'success': bool(result.success), 'checks': checks}
    return bool(result.success), failures, summary


def gate(df, layer):
    # Validate and STOP the pipeline if the data is not fit for the next stage.
    ok, failures, summary = validate(df, layer)
    if not ok:
        raise QualityGateFailed(layer, failures, summary)
    return summary

Overwriting /content/drive/MyDrive/sdaia_capstone/src/quality.py


## 4. The lineage emitter (OpenLineage)

`stage()` is a context manager: it emits **START** on entry, **COMPLETE** on a clean exit, and
**FAIL** with an `errorMessage` facet if the body raises. Wrapping every DAG task in it gives one
event stream for the whole pipeline.

Events go to a file transport (`reports/lineage_events.jsonl`) so no Marquez server is needed;
pointing `OPENLINEAGE_URL` at a collector is a one-line change.

In [6]:
%%writefile /content/drive/MyDrive/sdaia_capstone/src/lineage.py
# OpenLineage emission for every ShopSense pipeline stage.
import os
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

from openlineage.client import OpenLineageClient
from openlineage.client.transport.file import FileConfig, FileTransport
from openlineage.client.event_v2 import (RunEvent, RunState, Run, Job,
                                         InputDataset, OutputDataset)
from openlineage.client.uuid import generate_new_uuid
from openlineage.client.facet_v2 import job_type_job, error_message_run

NAMESPACE  = 'shopsense'
PRODUCER   = 'https://github.com/Bariah10/shopsense-data-platform'
EVENTS_LOG = os.environ.get(
    'SHOPSENSE_LINEAGE_LOG',
    '/content/drive/MyDrive/sdaia_capstone/reports/lineage_events.jsonl')


def _client():
    Path(EVENTS_LOG).parent.mkdir(parents=True, exist_ok=True)
    return OpenLineageClient(
        transport=FileTransport(FileConfig(log_file_path=EVENTS_LOG, append=True)))


def emit(state, job_name, run_id, inputs=(), outputs=(), error=None):
    facets = {}
    if error is not None:
        facets['errorMessage'] = error_message_run.ErrorMessageRunFacet(
            message=str(error)[:2000], programmingLanguage='PYTHON')
    event = RunEvent(
        eventType=state,
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=str(run_id), facets=facets),
        job=Job(namespace=NAMESPACE, name=job_name, facets={
            'jobType': job_type_job.JobTypeJobFacet(
                processingType='BATCH', integration='AIRFLOW', jobType='TASK')}),
        inputs=[InputDataset(namespace=NAMESPACE, name=n) for n in inputs],
        outputs=[OutputDataset(namespace=NAMESPACE, name=n) for n in outputs],
        producer=PRODUCER,
    )
    _client().emit(event)
    return event


@contextmanager
def stage(job_name, inputs=(), outputs=(), run_id=None):
    # START on entry, COMPLETE on success, FAIL on exception - then re-raise so
    # Airflow still marks the task failed and halts everything downstream.
    rid = run_id or generate_new_uuid()
    emit(RunState.START, job_name, rid, inputs, outputs)
    try:
        yield rid
    except Exception as exc:
        emit(RunState.FAIL, job_name, rid, inputs, outputs, error=exc)
        raise
    else:
        emit(RunState.COMPLETE, job_name, rid, inputs, outputs)


def read_events(path=None):
    import json
    p = Path(path or EVENTS_LOG)
    if not p.exists():
        return []
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]

Overwriting /content/drive/MyDrive/sdaia_capstone/src/lineage.py


In [7]:
import importlib, src.transforms, src.quality, src.lineage
for m in (src.transforms, src.quality, src.lineage):
    importlib.reload(m)
from src import transforms, quality, lineage
print('modules imported from', SRC)
print('suites available     :', list(quality.SUITES))

modules imported from /content/drive/MyDrive/sdaia_capstone/src
suites available     : ['bronze', 'silver']


## 5. Prove the gate - on clean data, then on bad data

### 5.1 Build Bronze from the real ingestion output, then gate it

In [8]:
info = transforms.ingest_to_bronze()
print('bronze built:', info)

bronze_df = transforms.read_bronze()
summary = quality.gate(bronze_df, 'bronze')          # does not raise -> gate PASSED
print(f"\nGATE PASSED on bronze: {summary['passed']}/{summary['evaluated']} expectations, "
      f"{summary['rows']} rows")
import pandas as pd
display(pd.DataFrame(summary['checks']))

bronze built: {'source_file': 'orders_valid_20260908T194115Z.jsonl', 'rows': 347, 'distinct_order_id': 322}


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpk22776mo' for ephemeral docs site


Calculating Metrics:   0%|          | 0/51 [00:00<?, ?it/s]


GATE PASSED on bronze: 8/8 expectations, 347 rows


,expectation,column,success,unexpected_count
0,expect_column_values_to_not_be_null,order_id,True,0
1,expect_column_values_to_not_be_null,customer_id,True,0
2,expect_column_values_to_be_between,quantity,True,0
3,expect_column_values_to_be_between,unit_price,True,0
4,expect_column_values_to_be_in_set,currency,True,0
5,expect_column_values_to_be_in_set,status,True,0
6,expect_column_values_to_be_in_set,category,True,0
7,expect_table_row_count_to_be_between,(table),True,0


### 5.2 Now corrupt the data on purpose

Four defects injected: a negative quantity, a null `customer_id`, an unsupported currency, and an
unknown category. A gate worth having must reject this.

In [9]:
corrupt = bronze_df.copy()
corrupt.loc[corrupt.index[0], 'quantity']    = -7
corrupt.loc[corrupt.index[1], 'customer_id'] = None
corrupt.loc[corrupt.index[2], 'currency']    = 'EUR'
corrupt.loc[corrupt.index[3], 'category']    = 'furniture'

try:
    quality.gate(corrupt, 'bronze')
    print('!! gate did not fire - that would be a bug')
except quality.QualityGateFailed as exc:
    print('GATE BLOCKED THE PIPELINE\n')
    print(exc)
    failed_summary = exc.summary

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp1x643fyf' for ephemeral docs site


Calculating Metrics:   0%|          | 0/51 [00:00<?, ?it/s]

GATE BLOCKED THE PIPELINE

Quality gate FAILED on bronze: 4/8 expectations failed
  - expect_column_values_to_not_be_null on customer_id: 1 unexpected value(s)
  - expect_column_values_to_be_between on quantity: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on currency: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on category: 1 unexpected value(s)


In [10]:
# Save the corrupted batch - notebook 05 points the DAG at it to prove the gate stops Airflow too
CORRUPT_FILE = PROJECT / 'data' / 'bronze_landing_corrupt.jsonl'
corrupt.to_json(CORRUPT_FILE, orient='records', lines=True, date_format='iso')
print('corrupted batch saved ->', CORRUPT_FILE)
print('rows:', len(corrupt))

corrupted batch saved -> /content/drive/MyDrive/sdaia_capstone/data/bronze_landing_corrupt.jsonl
rows: 347


### 5.3 The silver suite catches a different class of problem

In [11]:
silver_info = transforms.build_silver()
print('silver built:', silver_info)

silver_df = transforms.read_silver()
s = quality.gate(silver_df, 'silver')
print(f"\nGATE PASSED on silver: {s['passed']}/{s['evaluated']} expectations, {s['rows']} rows")

# A broken MERGE would duplicate the business key. Simulate that and watch the suite catch it.
dup = pd.concat([silver_df, silver_df.head(3)], ignore_index=True)
try:
    quality.gate(dup, 'silver')
except quality.QualityGateFailed as exc:
    print('\nDuplicate business keys rejected:\n', exc)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpbidbblwg' for ephemeral docs site


silver built: {'created': True, 'rows': 322, 'inserted': 322, 'updated': 0}


Calculating Metrics:   0%|          | 0/35 [00:00<?, ?it/s]

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpza3f3002' for ephemeral docs site



GATE PASSED on silver: 6/6 expectations, 322 rows


Calculating Metrics:   0%|          | 0/35 [00:00<?, ?it/s]


Duplicate business keys rejected:
 Quality gate FAILED on silver: 1/6 expectations failed
  - expect_column_values_to_be_unique on order_id: 6 unexpected value(s)


## 6. Prove the lineage events

In [12]:
from openlineage.client.event_v2 import RunState

EVENTS = REPORTS / 'lineage_events.jsonl'
if EVENTS.exists():
    EVENTS.unlink()          # start this demonstration from a clean file

# A stage that succeeds -> START then COMPLETE
with lineage.stage('build_silver_demo',
                   inputs=['lakehouse_dag.bronze.orders'],
                   outputs=['lakehouse_dag.silver.orders']):
    _ = transforms.read_silver()

# A stage that fails -> START then FAIL, carrying the reason
try:
    with lineage.stage('quality_gate_bronze_demo',
                       inputs=['lakehouse_dag.bronze.orders']):
        quality.gate(corrupt, 'bronze')
except quality.QualityGateFailed:
    print('stage failed as designed - FAIL event emitted\n')

events = lineage.read_events()
print(f'{len(events)} events in {EVENTS.name}\n')
for e in events:
    print(f"  {e['eventType']:9} {e['job']['name']:26} run={e['run']['runId'][-12:]}")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpxoz_atgb' for ephemeral docs site


Calculating Metrics:   0%|          | 0/51 [00:00<?, ?it/s]

stage failed as designed - FAIL event emitted

4 events in lineage_events.jsonl

  START     build_silver_demo          run=7c6f018a61fb
  COMPLETE  build_silver_demo          run=7c6f018a61fb
  START     quality_gate_bronze_demo   run=6aa40f80fc8e
  FAIL      quality_gate_bronze_demo   run=6aa40f80fc8e


In [13]:
fail_event = [e for e in events if e['eventType'] == 'FAIL'][0]
print('FAIL event, error facet:\n')
print(fail_event['run']['facets']['errorMessage']['message'][:600])
print('\ninputs :', [d['name'] for d in fail_event['inputs']])
print('job type facet:', fail_event['job']['facets']['jobType'])

FAIL event, error facet:

Quality gate FAILED on bronze: 4/8 expectations failed
  - expect_column_values_to_not_be_null on customer_id: 1 unexpected value(s)
  - expect_column_values_to_be_between on quantity: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on currency: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on category: 1 unexpected value(s)

inputs : ['lakehouse_dag.bronze.orders']
job type facet: {'_producer': 'https://github.com/OpenLineage/OpenLineage/tree/1.53.0/client/python', '_schemaURL': 'https://openlineage.io/spec/facets/2-0-4/JobTypeJobFacet.json#/$defs/JobTypeJobFacet', 'integration': 'AIRFLOW', 'jobType': 'TASK', 'processingType': 'BATCH'}


## 7. Stage report

In [14]:
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report = {
    'run_id': run_id,
    'stage': 'quality_and_lineage',
    'libraries': {'great_expectations': gx.__version__,
                  'openlineage_python': '1.53.0',
                  'deltalake': deltalake.__version__},
    'suites': {'bronze': len(quality.bronze_expectations()),
               'silver': len(quality.silver_expectations())},
    'bronze_gate': {'rows': summary['rows'], 'passed': summary['passed'],
                    'failed': summary['failed'], 'success': summary['success']},
    'bronze_gate_on_corrupt_data': {'failed_expectations': failed_summary['failed'],
                                    'success': failed_summary['success']},
    'silver_gate': {'rows': s['rows'], 'passed': s['passed'], 'failed': s['failed']},
    'lineage_events_file': str(EVENTS),
    'lineage_event_types': sorted({e['eventType'] for e in events}),
    'modules_written': [p.name for p in sorted(SRC.glob('*.py'))],
    'finished_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
out = REPORTS / f'quality_lineage_report_{run_id}.json'
out.write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
print('\nsaved ->', out)

{
  "run_id": "20260909T161417Z",
  "stage": "quality_and_lineage",
  "libraries": {
    "great_expectations": "1.22.0",
    "openlineage_python": "1.53.0",
    "deltalake": "1.6.3"
  },
  "suites": {
    "bronze": 8,
    "silver": 6
  },
  "bronze_gate": {
    "rows": 347,
    "passed": 8,
    "failed": 0,
    "success": true
  },
  "bronze_gate_on_corrupt_data": {
    "failed_expectations": 4,
    "success": false
  },
  "silver_gate": {
    "rows": 322,
    "passed": 6,
    "failed": 0
  },
  "lineage_events_file": "/content/drive/MyDrive/sdaia_capstone/reports/lineage_events.jsonl",
  "lineage_event_types": [
    "COMPLETE",
    "FAIL",
    "START"
  ],
  "modules_written": [
    "__init__.py",
    "lineage.py",
    "quality.py",
    "transforms.py"
  ],
  "finished_at": "2026-09-09T16:14:17.262631+00:00"
}

saved -> /content/drive/MyDrive/sdaia_capstone/reports/quality_lineage_report_20260909T161417Z.json


## What notebook 05 picks up

```
MyDrive/sdaia_capstone/
  src/
    transforms.py   <- the medallion stages
    quality.py      <- gate() raises QualityGateFailed
    lineage.py      <- stage() emits START / COMPLETE / FAIL
  data/bronze_landing_corrupt.jsonl   <- used to make the DAG fail on purpose
  reports/lineage_events.jsonl
```

Save with `File -> Save`, then push this notebook and `src/` to GitHub.